# Model Comparison
This is a set of "superficial" tests to compare the performance of multiple OpenAI models on running different tasks for possible RAG stages. The results are shown in the tables at the beginning of each section. Please note that tests are not extensive or statistically significant but rather a quick comparison based on a few runs to choose the simplest model for each task.

## Setup

In [1]:
import fs from 'node:fs';
import path from 'node:path';
import { ChatOpenAI } from '@langchain/openai';
import { OPENAI_API_KEY } from './src/lib/vars.mjs';
import * as z from 'zod';

function logFile(logContent, fileName = 'jupyter.md') {
  const logFileName = `logs/${fileName}`;
  const logDir = path.dirname(logFileName);
  if (!fs.existsSync(logDir)) {
    fs.mkdirSync(logDir, { recursive: true });
  }
  fs.writeFileSync(logFileName, '```markdown\n' + logContent + '\n```', 'utf8');
  return logContent;
}

const gpt5 = new ChatOpenAI({
  modelName: 'gpt-5',
  apiKey: OPENAI_API_KEY,
  temperature: 1,
});
const gpt4oMini = new ChatOpenAI({
  modelName: 'gpt-4o-mini',
  apiKey: OPENAI_API_KEY,
  temperature: 0.1,
});
const gpt5Nano = new ChatOpenAI({
  modelName: 'gpt-5-nano',
  apiKey: OPENAI_API_KEY,
  temperature: 1,
});
const gpt41Nano = new ChatOpenAI({
  modelName: 'gpt-4.1-nano',
  apiKey: OPENAI_API_KEY,
  temperature: 0.1,
});

async function queryAllModels(prompt, callFunction) {
  console.log(`Started: ${new Date()}\n`);
  return Promise.all([
    callFunction(prompt, gpt4oMini),
    callFunction(prompt, gpt41Nano),
    callFunction(prompt, gpt5Nano),
    // callFunction(prompt, gpt5),
  ]);
}

## Rewrite Initial Query


### Sanitize Query

**Results:** 

All models performed satisfactory with similar times.

In [2]:
import { sanitizeQuery } from './src/lib/rag.mjs';

const redditQuestion = `Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if this was too long or boring to read and i cannot explain myself completely bc english isnt my first language, but help what should we do? (Ask any question you need to ask)`;

async function sanitize(question, model) {
  const { output: query } = await sanitizeQuery(question, model);
  console.log(`**${model.model}** – ${new Date()}:\n${query}\n`);
  return query;
}

await queryAllModels(redditQuestion, sanitize);


Started: Tue Aug 26 2025 13:32:49 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** – Tue Aug 26 2025 13:32:52 GMT-0400 (Eastern Daylight Time):
I am an exchange student planning to study in Canada for a year, but I am concerned about my visa application. We submitted the visa request around June 15, and were initially told it would take about 6 weeks, but the processing time has now changed to 10 weeks. I need to leave by August 29 to start school on September 2, and I am worried that my visa will not arrive in time. We are considering applying for a tourist visa to enter Canada while waiting for my study visa, but I am concerned that this may affect my ability to return to Canada after Christmas. What are my options regarding the visa situation, and what should I do to ensure I can study in Canada without issues?

**gpt-4o-mini** – Tue Aug 26 2025 13:32:52 GMT-0400 (Eastern Daylight Time):
What are the options for an exchange student who is applying for a study permit to Canada but 

[
  "What are the options for an exchange student who is applying for a study permit to Canada but is facing delays in the visa processing time? I applied for the visa around June 15th, and while I was initially told it would take about 6 weeks, it has now been extended to 10 weeks. I need to leave by August 29th to start school on September 2nd, but I am concerned that my visa may not arrive in time. If I travel to Canada on a tourist visa, will that affect my ability to return to Canada after Christmas once my study permit is issued? What steps should I take in this situation?",
  "I am an exchange student planning to study in Canada for a year, but I am concerned about my visa application. We submitted the visa request around June 15, and were initially told it would take about 6 weeks, but the processing time has now changed to 10 weeks. I need to leave by August 29 to start school on September 2, and I am worried that my visa will not arrive in time. We are considering applying fo

### Query Extraction

#### Decomposing Main Query

#### Test Results

| Model          | Response Time | Performance | Notes                                                   |
| -------------- | ------------- | ----------- | ------------------------------------------------------- |
| 🥇 GPT-4o-mini | < 5 sec       | Good        |                                                         |
| GPT-4.1-nano   | < 5 sec       | Good        |                                                         |
| GPT-5-nano     | < 30 sec      | Mediocre    | Questions need further break down.                      |
| GPT-5          | > 1 min       | Ok          | Questions well-thought but must be broken down further. |


In [3]:
const compoundQuestion = `Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if this was too long or boring to read and i cannot explain myself completely bc english isnt my first language, but help what should we do? (Ask any question you need to ask)`;

const questionExtractionPrompt = `You are an assistant that prepares user queries for a Retrieval-Augmented Generation (RAG) system about Canadian immigration rules.
The user may provide a long, informal story or question. Your task is:
1. Identify all explicit and implicit questions they are asking.  
2. Rewrite each one as a clear, self-contained question that could be answered directly from IRCC documentation.  
3. Condense the result into the *smallest possible set of non-overlapping, atomic questions* that fully capture the user’s intent.  
4. Eliminate redundancy — avoid rephrasing the same issue multiple times.  
5. Do not provide answers — only the minimal list of questions.

**User question:**

\`\`\`
${compoundQuestion}
\`\`\`
`;

async function extractQuestions(q, model) {
  const response = await model
    .withStructuredOutput(
      z.object({
        questions: z
          .array(z.string())
          .describe('A list of questions derived from the user query, sorted by relevance top to bottom.'),
      })
    )
    .invoke(q);
  const content = response.questions.map((q) => `\t- ${q}`).join('\n');
  console.log(`**${model.model}** ${new Date()}:\n${content}\n`);
  return response;
}

const extractedQuestions = await queryAllModels(questionExtractionPrompt, extractQuestions);


Started: Tue Aug 26 2025 13:33:00 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** Tue Aug 26 2025 13:33:01 GMT-0400 (Eastern Daylight Time):
	- Can I enter Canada as a student with a valid study permit obtained before my planned travel date?
	- Is it possible to enter Canada as a tourist and then switch to a student status after arrival?
	- What are the implications of entering Canada on a tourist visa and later changing to a student visa?
	- Can I leave Canada during my studies and re-enter with a tourist visa, and will this affect my student status?
	- What are the risks of traveling to Canada on a tourist visa if my study permit has not yet been approved?
	- Is there a way to expedite my study permit application to ensure I arrive before my school starts?
	- What are the options if my study permit is delayed beyond my planned arrival date?

**gpt-4o-mini** Tue Aug 26 2025 13:33:02 GMT-0400 (Eastern Daylight Time):
	- What are the current processing times for a study permit applic

#### Identifying Key Questions

##### Test Summary

| Model           | Response Time | Performance | Notes                                  |
| --------------- | ------------- | ----------- | -------------------------------------- |
| 🥇 GPT-4.1-nano | < 5 sec       | Good        | Correctly identified the key question. |
| GPT-4o-mini     | < 5 sec       | Good        | Included some secondary questions.     |
| GPT-5-nano      | < 30 sec      | Mediocre    | Included the most questions.           |
| GPT-5           | < 30 sec      | Ok          | Included some secondary questions.     |


In [4]:

const mdExtractedQuestions = extractedQuestions[1].questions.map(q => `\t- ${q}`).join('\n');
const questionDiscriminationPrompt = `You are helping prepare user queries for a Retrieval-Augmented Generation (RAG) system about Canadian immigration.
Input: a list of atomic questions generated from a user’s long query.
Task:
1. Identify the key question(s) that directly capture the user’s main intent.  
   - Keep only the questions that must be answered to resolve the user’s core concern.  
   - Discard questions that are secondary, conditional, or only relevant as follow-ups.
2. Output only the minimal set of key questions, without explanation, ranked by relevance to the user query.
Important: The result should be as short as possible while still fully representing the original user’s primary intent.

User query:

\`\`\`
${compoundQuestion}
\`\`\`

List of questions:
\`\`\`
${mdExtractedQuestions}
\`\`\`
`

async function discriminateQuestions(q, model) {
  const response = await model
    .withStructuredOutput(
      z.object({
        questions: z.array(z.string()).describe('A list of key questions derived from the user query'),
      })
    )
    .invoke(q);
  const content = response.questions.map(q => `\t- ${q}`).join('\n');
  console.log(`**${model.model}** ${new Date()}:\n${content}\n`);
  return response;
}

console.log(`\n\nDiscriminating questions for: "${compoundQuestion}"\n`);
console.log(`Extracted questions:\n${mdExtractedQuestions}\n`);
await queryAllModels(questionDiscriminationPrompt, discriminateQuestions);



Discriminating questions for: "Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if th

[
  {
    questions: [
      "Is it possible to enter Canada as a tourist and then switch to a student status after arrival?",
      "What are the implications of entering Canada on a tourist visa and later changing to a student visa?",
      "What are the risks of traveling to Canada on a tourist visa if my study permit has not yet been approved?",
      "Is there a way to expedite my study permit application to ensure I arrive before my school starts?"
    ]
  },
  {
    questions: [
      "Can I enter Canada as a student with a valid study permit obtained before my planned travel date?",
      "Is it possible to enter Canada as a tourist and then switch to a student status after arrival?",
      "What are the implications of entering Canada on a tourist visa and later changing to a student visa?",
      "Can I leave Canada during my studies and re-enter with a tourist visa, and will this affect my student status?",
      "What are the risks of traveling to Canada on a tourist visa i

## Vector Search

### Retrieval
(No LLMs involved in this stage, but this stage is needed for the following tests).

In [3]:
import { vectorSearch, chunksToMarkdown } from './src/lib/vector-search.mjs';
const retrieveQuery = 'Can I enter Canada on a tourist visa while waiting for my study permit?'
const chunks = await vectorSearch(retrieveQuery);
logFile(chunksToMarkdown(chunks), 'chunks.md');
chunks

[
  {
    text: "# Study permit\n" +
      "\n" +
      "\\[...\\]\n" +
      "\n" +
      "* * * \n" +
      "\n" +
      "On the application form, select both\n" +
      "\n" +
      "*   an initial study permit or extension of study permit and\n" +
      "*   restoration of temporary resident status as a student\n" +
      "\n" +
      "Find out if you’re eligible and how to [restore your student status](/en/immigration-refugees-citizenship/services/study-canada/extend-study-permit/expired-permit.html).\n" +
      "\n" +
      "## Apply at a port of entry\n" +
      "\n" +
      "### When you arrive at the port of entry\n" +
      "\n" +
      "Tell the officer that you want a study permit. The officer will check\n" +
      "\n" +
      "*   your passport or other travel document\n" +
      "*   that you meet the eligibility requirements and\n" +
      "*   that your medical certificate is valid, if you need one\n" +
      "\n" +
      "If you’re eligible for a study permit, the off

### Reference Discrimination

#### Test Summary

| Model          | Response Time | Performance | Notes                                 |
| -------------- | ------------- | ----------- | ------------------------------------- |
| 🥇 GPT-4o-mini | < 5 sec       | Good        | Picked the most chunks, all relevant. |
| GPT-4.1        | < 5 sec       | Ok          |                                       |
| GPT-5          | < 1 min       | Ok          |                                       |
| GPT-5-nano     | < 30 sec      | Mediocre    | Missed a highly relevant chunk.       |


In [ ]:
const chunkDiscriminationPrompt = `I'll give you a markdown content with a list of results from a vector search for a question.
Select only the references that help to answer th question.
Pay attention to the header of the top-most header in each reference to identify if it's related to the topic we want to answer; discard it if it is not.
Return an array containing the selected references' numbers.

**Question:** ${retrieveQuery}

**Chunks:**

\`\`\`markdown
${chunksToMarkdown(chunks)}
\`\`\`
`;

async function evaluateFollowUp(prompt, model) {
  const { references: referenceIndexes } = await model
  .withStructuredOutput(
    z.object({
      references: z.array(z.number()).describe('An array of numbers representing the selected references from the chunks'),
    })
  )
  .invoke(prompt);

  console.log(`**${model.model}** – ${new Date()}: ${JSON.stringify(referenceIndexes)}`);
  return referenceIndexes;
}

await queryAllModels(chunkDiscriminationPrompt, evaluateFollowUp);

Started: Tue Aug 26 2025 13:34:42 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** – Tue Aug 26 2025 13:34:42 GMT-0400 (Eastern Daylight Time): [2]
**gpt-4o-mini** – Tue Aug 26 2025 13:34:43 GMT-0400 (Eastern Daylight Time): [2,5]
**gpt-5-nano** – Tue Aug 26 2025 13:34:52 GMT-0400 (Eastern Daylight Time): [2,5]


## Answering

### Generating Answer

#### Test Summary

| Model       | Response Time | Performance | Notes                             |
| ----------- | ------------- | ----------- | --------------------------------- |
| GPT-4.1     | < 5 sec       | Good        | Good explanations.                |
| GPT-4o-mini | < 5 sec       | Ok          | Somhow "superficial".             |
| GPT-5-nano  | < 30 sec      | Good        | Good structure. Too many details. |
| GPT-5       | < 1 min       | Ok          | Somhow "superficial".             |


In [4]:
import { generateAnswer } from './src/lib/rag.mjs';

async function generateRAGAnswer(query, model) {
  const references = chunksToMarkdown(chunks);
  const response = await generateAnswer(query, references, [], model);
  console.log(
    `**${model.model}** – ${new Date()}:\n${response}\n\n* * *\n`
  );
  return response;
}

console.log(`"${retrieveQuery}"\n`);
const generatedResponses = await queryAllModels(retrieveQuery, generateRAGAnswer);

"Can I enter Canada on a tourist visa while waiting for my study permit?"

Started: Tue Aug 26 2025 13:38:57 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** – Tue Aug 26 2025 13:39:01 GMT-0400 (Eastern Daylight Time):
Based on the IRCC documentation, you generally cannot enter Canada on a tourist visa while waiting for your study permit to be approved.

If you are outside Canada and plan to come as a visitor, IRCC allows entry as a visitor if your study permit application is still in process, but you cannot study until your study permit is issued. The decision to admit you as a visitor or a student at the port of entry is made by the immigration officer, and there is a possibility you may not be allowed to enter as a visitor if the officer deems it appropriate [[1](https://www.canada.ca/en/immigration-refugees-citizenship/services/study-canada/study-permit/apply.html)].

If you are already in Canada and your study permit application is pending, you cannot simply switch to a tourist 